# 🏝️ Spike Island — Interactive Walkthrough

A comprehensive, hands-on tour of the **Spike Island** neurotech toolkit.
Each section demonstrates one module with real data, visualizations, and
explanations of the underlying neurophysiology.

**Prerequisites:** `pip install -e .` (installs numpy, matplotlib, scipy)

---

## 0. Setup

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
matplotlib.style.use('seaborn-v0_8-whitegrid')

SEED = 42
rng = np.random.default_rng(SEED)

print(f"Spike Island version: {spike_island.__version__}")
print(f"NumPy: {np.__version__}, Matplotlib: {matplotlib.__version__}")

## 1. Spike Train Simulators (Day 1)

Four firing patterns model different neural behaviors:

| Pattern | Biology | CV |
|---|---|---|
| **Poisson** | Memoryless, random firing | ~1.0 |
| **Refractory Poisson** | Absolute refractory period enforced | 0.6–0.9 |
| **Bursty Poisson** | Bursts of spikes on background | 1.5–2.5 |
| **Rhythmic** | Regular oscillation with jitter | 0.0–0.5 |

In [ ]:
from spike_island.simulators import (
    poisson_spikes, refractory_poisson,
    bursty_poisson, rhythmic_spikes,
)

duration_ms = 2000.0

spike_poisson = poisson_spikes(rate_hz=50.0, duration_ms=duration_ms, seed=SEED)
spike_refractory = refractory_poisson(rate_hz=50.0, duration_ms=duration_ms,
                                        refractory_ms=2.0, seed=SEED)
spike_bursty = bursty_poisson(background_rate_hz=30.0, burst_rate=10.0,
                                 burst_size_mean=5, burst_size_std=2.0,
                                 intra_burst_isi_ms=3.0, duration_ms=duration_ms, seed=SEED)
spike_rhythmic = rhythmic_spikes(rate_hz=30.0, duration_ms=duration_ms,
                                   jitter_sd_ms=1.0, seed=SEED)

# Plot raster for each pattern
fig, axes = plt.subplots(4, 1, figsize=(12, 8), sharex=True)
patterns = [
    ("Poisson (CV≈1.0)", spike_poisson),
    ("Refractory Poisson (CV<1)", spike_refractory),
    ("Bursty Poisson (CV>1)", spike_bursty),
    ("Rhythmic (CV≈0)", spike_rhythmic),
]
for ax, (label, spikes) in zip(axes, patterns):
    for t in spikes:
        ax.eventplot([t], lineoffsets=np.zeros_like([t]) + 0.5,
                     color='steelblue', linewidth=1.5)
    ax.set_ylabel(label)
    ax.set_ylim(0, 1.5)
    ax.set_yticks([])
axes[-1].set_xlabel('Time (ms)')
plt.tight_layout()
plt.show()

print(f"Poisson: {len(spike_poisson)} spikes")
print(f"Refractory: {len(spike_refractory)} spikes")
print(f"Bursty: {len(spike_bursty)} spikes")
print(f"Rhythmic: {len(spike_rhythmic)} spikes")

## 2. Analysis Metrics (Day 2)

Key neurophysiology metrics:

- **Firing rate** — spikes per second
- **ISI histogram** — distribution of inter-spike intervals
- **Coefficient of variation (CV)** — std(ISI) / mean(ISI); ~1 for Poisson
- **Autocorrelogram** — spike timing correlation with itself

In [ ]:
from spike_island.analysis import (
    analyze, coefficient_of_variation,
    autocorrelogram, isi_histogram, mean_firing_rate,
)

# Analyze the Poisson train
analysis = analyze(spike_poisson, duration_ms, name="Poisson")

print(f"Firing rate: {analysis['firing_rate']:.1f} Hz")
print(f"CV: {analysis['cv']:.3f} (Poisson ≈ 1.0)")
print(f"Mean ISI: {analysis['mean_isi_ms']:.1f} ms")
print(f"Median ISI: {analysis['median_isi_ms']:.1f} ms")

# Compare CV across all four patterns
fig, ax = plt.subplots(figsize=(10, 5))
cv_values = [
    coefficient_of_variation(spike_poisson),
    coefficient_of_variation(spike_refractory),
    coefficient_of_variation(spike_bursty),
    coefficient_of_variation(spike_rhythmic),
]
labels = ['Poisson', 'Refractory', 'Bursty', 'Rhythmic']
colors = ['#1f77b4', '#2ca02c', '#d62728', '#ff7f0e']
bars = ax.bar(labels, cv_values, color=colors, width=0.6)
ax.axhline(y=1.0, color='gray', linestyle='--', label='Poisson baseline (CV=1)')
for bar, val in zip(bars, cv_values):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.02,
            f'{val:.2f}', ha='center', fontsize=11)
ax.set_ylabel('Coefficient of Variation')
ax.set_title('CV across firing patterns')
ax.legend()
plt.tight_layout()
plt.show()

# ISI histogram + autocorrelogram
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

isi_bins, isi_counts = isi_histogram(spike_poisson, bin_width_ms=5.0)
axes[0].bar(isi_bins, isi_counts, width=5.0, color='steelblue', alpha=0.8)
axes[0].set_xlabel('ISI (ms)')
axes[0].set_ylabel('Count')
axes[0].set_title('ISI Histogram (Poisson)')

acg_lags, acg_counts = autocorrelogram(spike_poisson, bin_width_ms=5.0)
axes[1].bar(acg_lags, acg_counts, width=5.0, color='coral', alpha=0.8)
axes[1].set_xlabel('Lag (ms)')
axes[1].set_ylabel('Count')
axes[1].set_title('Autocorrelogram (Poisson)')
plt.tight_layout()
plt.show()

## 3. Waveform Visualization (Day 3)

Convert discrete spike times into continuous-time voltage traces by convolving
with an action potential (AP) template. Add noise to simulate real recordings.

In [ ]:
from spike_island.waveforms import (
    generate_ap_template, spikes_to_waveform,
    waveform_statistics, detect_spikes_from_waveform,
)

# Generate AP template
tpl_t, tpl = generate_ap_template(amplitude_mv=-2.0)

# Build continuous waveform from Poisson spikes
times, voltage = spikes_to_waveform(
    spike_poisson, tpl,
    template_dt=0.1, duration_ms=duration_ms,
    sampling_hz=10_000.0, noise_std_mv=0.05,
    noise_type='gaussian', seed=SEED,
)

stats = waveform_statistics(voltage, times)
print(f"Samples: {stats['samples']}")
print(f"Duration: {stats['duration_ms']:.0f} ms")
print(f"Peak-to-peak: {stats['peak_to_peak_mv']:.2f} mV")
print(f"SNR: {stats['snr_db']:.1f} dB")

# Plot template + waveform
fig, axes = plt.subplots(2, 1, figsize=(12, 8), sharex=False)

axes[0].plot(tpl_t, tpl, linewidth=2, color='steelblue')
axes[0].set_ylabel('Voltage (mV)')
axes[0].set_xlabel('Time (ms)')
axes[0].set_title('Action Potential Template')
axes[0].axhline(y=0, color='gray', linewidth=0.5)

axes[1].plot(times, voltage, linewidth=0.5, color='coral', alpha=0.8)
axes[1].set_ylabel('Voltage (mV)')
axes[1].set_xlabel('Time (ms)')
axes[1].set_title(f'Waveform ({len(spike_poisson)} spikes, Gaussian noise)')
axes[1].axhline(y=0, color='gray', linewidth=0.5)
plt.tight_layout()
plt.show()

# Detect spikes from the waveform
detected_indices = detect_spikes_from_waveform(
    voltage, times, threshold_mv=-0.5, refractory_ms=2.0,
)
print(f"Detected {len(detected_indices)} spikes from waveform (ground truth: {len(spike_poisson)})")

## 4. Spike Sorting (Day 4)

Template-matching spike sorter: given a multi-unit recording and known templates,
detect spike events and assign them to neurons using matching pursuit.

In [ ]:
from spike_island.sorting import (
    generate_ground_truth, contaminate_recording,
    template_sort, evaluate_sorting, print_sorting_report,
)

# Generate ground truth: 3 neurons with different rates
true_spikes, true_assignments = generate_ground_truth(
    rates_hz=[40.0, 25.0, 15.0],
    duration_ms=duration_ms, seed=SEED,
)
print(f"Ground truth: {len(true_assignments)} total spikes across 3 neurons")

# Create templates for each neuron
templates = [
    np.array([-5.0, 0.0, 2.5, -0.3]),
    np.array([-3.0, 0.5, 0.0]),
    np.array([-4.0, 1.0, -0.5, 0.2]),
]

# Build noisy multi-unit recording
recording, assignments = contaminate_recording(
    spike_times_list=true_spikes,
    templates=templates,
    noise_std=0.3, sampling_hz=10_000.0,
    duration_ms=duration_ms, seed=SEED,
)

# Run template-matching sorter
detected, counts = template_sort(
    recording=recording, templates=templates,
    dt_ms=0.1, threshold=0.3,
    refractory_ms=2.0, time_window_ms=1.0,
    max_iter=50,
)

print(f"Detected {sum(counts.values())} spikes across {len(counts)} neurons")
for nid, cnt in counts.items():
    print(f"  Neuron {nid}: {cnt} spikes")

# Evaluate sorting quality
report = evaluate_sorting(
    true_spikes=true_assignments,
    detected_spikes=detected,
    merge_window_ms=0.5,
)
print_sorting_report(report)

# Plot recording with detected events
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(np.arange(len(recording)) * 0.1, recording, linewidth=0.3, color='gray', alpha=0.7)
for event in detected:
    color = {0: 'red', 1: 'blue', 2: 'green'}.get(event['neuron_id'], 'purple')
    ax.plot(event['time_ms'], 0, 'o', color=color, markersize=6, alpha=0.8)
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Voltage (a.u.)')
ax.set_title(f'Multi-unit recording with sorted spikes (N={sum(counts.values())})')
ax.set_xlim(0, duration_ms)
plt.tight_layout()
plt.show()

## 5. Wilson-Cowan Neural Oscillator (Day 5)

Mean-field model of coupled excitatory (E) and inhibitory (I) populations:

$$\tau_E \frac{dE}{dt} = -E + W_{EE} \phi(E - \theta_E) - W_{EI} \phi(I - \theta_I) + I_E$$
$$\tau_I \frac{dI}{dt} = -I + W_{IE} \phi(E - \theta_E) - W_{II} \phi(I - \theta_I) + I_I$$

Can exhibit: fixed points, bistability, or sustained oscillations.

In [ ]:
from spike_island.oscillator import (
    WCParams, simulate_wc, wc_fixed_points,
    wc_stability, wc_bifurcation_scan,
    wc_oscillation_metrics, wc_oscillatory,
)

# Use the oscillatory preset
params = wc_oscillatory()
print(f"Parameters: W_EE={params.w_EE}, W_EI={params.w_EI}, W_IE={params.w_IE}, W_II={params.w_II}")

# Simulate
t, traj = simulate_wc(params, t_max=3000.0, dt=0.5, seed=SEED)
E = traj[:, 0]
I = traj[:, 1]

# Plot time series
fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(t, E, linewidth=1.5, color='steelblue', label='Excitatory (E)')
ax.plot(t, I, linewidth=1.5, color='coral', label='Inhibitory (I)')
ax.set_xlabel('Time (ms)')
ax.set_ylabel('Population activity')
ax.set_title('Wilson-Cowan: E-I Oscillator')
ax.legend()
plt.tight_layout()
plt.show()

# Phase portrait
fig, ax = plt.subplots(figsize=(6, 6))
ax.plot(E, I, linewidth=0.8, color='purple', alpha=0.6)
ax.set_xlabel('E (excitatory)')
ax.set_ylabel('I (inhibitory)')
ax.set_title('Phase Portrait (E vs I)')
plt.tight_layout()
plt.show()

# Fixed points & stability
fps = wc_fixed_points(params)
print(f"Found {len(fps)} fixed point(s):")
for i, fp in enumerate(fps):
    stab = wc_stability(fp, params)
    print(f"  FP {i}: E={fp[0]:.3f}, I={fp[1]:.3f}, type={stab['type']}, eigenvalues={stab['eigenvalues']}")

# Oscillation metrics
metrics = wc_oscillation_metrics(t, traj, transient_ms=1000.0)
print(f"\nOscillation metrics:")
print(f"  Oscillating: {metrics['oscillating']}")
if metrics['oscillating']:
    print(f"  Frequency: {metrics['frequency_hz']:.2f} Hz")
    print(f"  Period: {metrics['period_ms']:.1f} ms")
    print(f"  E amplitude: {metrics['e_amplitude']:.3f}")
    print(f"  Phase lag: {metrics['phase_lag_deg']:.1f}°")

# Bifurcation scan: vary W_EE
w_ee_values = np.linspace(0.4, 2.0, 60)
bifurcation = wc_bifurcation_scan(
    params, vary_param='w_EE', param_values=w_ee_values,
    t_max=5000.0, transient_ms=1000.0, dt=0.5, seed=SEED,
)
fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(w_ee_values, bifurcation[:, 1], 'o-', color='coral',
        markersize=4, label='I_min', alpha=0.8)
ax.plot(w_ee_values, bifurcation[:, 2], 'o-', color='steelblue',
        markersize=4, label='I_max', alpha=0.8)
ax.fill_between(w_ee_values, bifurcation[:, 1], bifurcation[:, 2],
                alpha=0.2, color='purple')
ax.set_xlabel('W_EE (excitatory self-coupling)')
ax.set_ylabel('I (inhibitory activity)')
ax.set_title('Bifurcation Diagram: varying W_EE')
ax.legend()
plt.tight_layout()
plt.show()

## 6. STDP Learning Engine (Day 6)

Spike-Timing Dependent Plasticity: synaptic weights change based on the
relative timing of pre- and post-synaptic spikes.

- **Causal** (pre before post) → LTP (weight increase)
- **Anti-causal** (post before pre) → LTD (weight decrease)

Learning rule: $\Delta w = A_+ e^{-\Delta t / \tau_+}$ (causal) or
$\Delta w = -A_- e^{\Delta t / \tau_-}$ (anti-causal)

In [ ]:
from spike_island.stdp import (
    STDPParams, Synapse, STDPNetwork,
    stdp_weight_update, plot_stdp_learning_window,
)

# Define STDP parameters
params = STDPParams(
    a_plus=0.01, a_minus=0.012,
    tau_plus=20.0, tau_minus=20.0,
    w_min=0.0, w_max=1.0, history_length=50.0,
)

# Plot the learning window
fig, ax = plt.subplots(figsize=(10, 5))
plot_stdp_learning_window(params, ax=ax)
ax.set_title('STDP Learning Window')
plt.tight_layout()
plt.show()

# Single synapse: causal pairing (LTP)
syn = Synapse(weight=0.5, params=params)
n_trials = 50
weights_causal = []
for _ in range(n_trials):
    t_pre = rng.uniform(0.0, 10.0)
    t_post = t_pre + rng.uniform(1.0, 5.0)  # post after pre → LTP
    syn.record_pre_spike(t_pre)
    syn.record_post_spike(t_post)
    weights_causal.append(syn.weight)

# Single synapse: anti-causal pairing (LTD)
syn2 = Synapse(weight=0.5, params=params)
weights_anticausal = []
for _ in range(n_trials):
    t_post = rng.uniform(0.0, 10.0)
    t_pre = t_post + rng.uniform(1.0, 5.0)  # pre after post → LTD
    syn2.record_pre_spike(t_pre)
    syn2.record_post_spike(t_post)
    weights_anticausal.append(syn2.weight)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(range(n_trials), weights_causal, 'o-', color='steelblue',
        markersize=4, label=f'Causal (LTP) → {syn.weight:.3f}')
ax.plot(range(n_trials), weights_anticausal, 's-', color='coral',
        markersize=4, label=f'Anti-causal (LTD) → {syn2.weight:.3f}')
ax.axhline(y=0.5, color='gray', linestyle='--', alpha=0.5, label='Initial weight')
ax.set_xlabel('Trial')
ax.set_ylabel('Synaptic weight')
ax.set_title('STDP: Single Synapse Weight Evolution')
ax.legend()
plt.tight_layout()
plt.show()

# Network-level STDP
net = STDPNetwork(
    n_pre=10, n_post=10,
    params=params, w_init=0.5, history_length=50.0,
)
# Generate random spike events
for _ in range(500):
    pre_neuron = rng.integers(0, 10)
    pre_t = rng.uniform(0.0, 100.0)
    net.record_pre_spike(int(pre_neuron), pre_t)
    post_neuron = rng.integers(0, 10)
    post_t = rng.uniform(0.0, 100.0)
    net.record_post_spike(int(post_neuron), post_t)

stats = net.get_stats()
print(f"\nNetwork stats after 500 spike events:")
print(f"  Mean weight: {stats['mean_weight']:.3f}")
print(f"  Std weight: {stats['std_weight']:.3f}")
print(f"  Min weight: {stats['min_weight']:.3f}")
print(f"  Max weight: {stats['max_weight']:.3f}")

# Heatmap of weight matrix
fig, ax = plt.subplots(figsize=(7, 6))
im = ax.imshow(net.weights, cmap='RdYlBu_r', vmin=0.0, vmax=1.0, aspect='auto')
ax.set_xlabel('Post-synaptic neuron')
ax.set_ylabel('Pre-synaptic neuron')
ax.set_title('STDP Network: Weight Matrix (10×10)')
plt.colorbar(im, ax=ax, label='Weight')
plt.tight_layout()
plt.show()

## 7. Full Pipeline (Day 7)

The pipeline orchestrates all modules into a unified workflow:
1. Generate spike trains → 2. Analyze → 3. Build waveforms →
4. Sort spikes → 5. Simulate Wilson-Cowan → 6. Learn via STDP

In [ ]:
from spike_island.pipeline import PipelineConfig, run_and_report

config = PipelineConfig(
    duration_ms=2000.0,
    sampling_hz=10_000.0,
    seed=SEED,
    output_dir='plots',
    wc_t_max=2000.0,
    stdp_steps=30,
)

result = run_and_report(config)

print(f"\nPipeline completed in {result.total_elapsed_ms:.1f} ms")
print(f"Stages executed: {len(result.stages)}")
for stage in result.stages:
    print(f"  {stage.name}: {stage.elapsed_ms:.1f} ms")

## 8. Performance Benchmarks (Day 8)

Systematic benchmarks for every module — timing, memory, and throughput.

In [ ]:
from spike_island.benchmarks import (
    benchmark_simulators, benchmark_analysis,
    benchmark_waveforms, benchmark_sorting,
    benchmark_oscillator, benchmark_stdp,
    print_benchmark_report,
)

# Run a subset of benchmarks (excluding the slow pipeline benchmark)
all_results = []
for bench_func, name in [
    (benchmark_simulators, "Simulators"),
    (benchmark_analysis, "Analysis"),
    (benchmark_waveforms, "Waveforms"),
    (benchmark_sorting, "Sorting"),
    (benchmark_oscillator, "Oscillator"),
    (benchmark_stdp, "STDP"),
]:
    print(f"Running {name} benchmarks...")
    results = bench_func()
    all_results.extend(results)

print(f"\nTotal benchmarks run: {len(all_results)}")

# Summary table
from spike_island.benchmarks import BenchmarkSuite
suite = BenchmarkSuite()
for r in all_results:
    suite.add(r)
suite.total_elapsed_ms = sum(r.elapsed_ms for r in all_results)

report = print_benchmark_report(suite)

## Summary

This notebook demonstrated all 8 modules of Spike Island:

| # | Module | What it does |
|---|---|---|
| 1 | **Simulators** | Generate spike trains (Poisson, refractory, bursty, rhythmic) |
| 2 | **Analysis** | Compute ISI, CV, autocorrelogram, firing rate |
| 3 | **Waveforms** | Build continuous-time AP waveforms with noise |
| 4 | **Sorting** | Template-matching spike sorter for multi-unit data |
| 5 | **Wilson-Cowan** | E-I neural mass oscillator with bifurcation analysis |
| 6 | **STDP** | Spike-timing dependent plasticity learning |
| 7 | **Pipeline** | Full integrated workflow with visualization |
| 8 | **Benchmarks** | Performance profiling across all modules |

🏝️ *Spike Island — built for learning neurotech fundamentals.*